In [ ]:
# -*- coding: utf-8 -*-
"""
FIN 285A – Spring 2026  |  Research Project Part 2
Improved All Weather ETF via Tracking Error Minimization

Builds a disaggregated version of the rescaled (200% leverage) Bridgewater
All Weather benchmark by splitting global equities into regional components,
bonds into sub-types, and commodities into individual commodities, then solving
for the weights that minimize in-sample tracking error against the benchmark.

Training: 2014-01-01 to 2018-12-31  (5 years — per project instruction)
Testing:  2019-01-01 to end of available data

Sensitivity analysis (Section 10) tests longer windows (up to 7 years)
per slide 12: "change the training/test data set length somewhat."

Fund universe (20 assets):
    Equity sleeve (7):  VTI, HEDJ, VWO, QQQ, IWM, EWJ, INDA
    Bond sleeve (5):    AGG, BNDX, TLT, LQD, HYG
    TIPS sleeve (2):    TIP, STPZ
    Commodity sleeve (6): GLD, USO, CPER, SLV, DBA, UNG

Benchmark (rescaled 200% All Weather — SAME proxies as Part 1):
    AGG  – Global nominal bonds    76.12%
    ACWI – Global equities         48.49%
    GSG  – Commodities             40.66%
    TIP  – Inflation linked bonds  34.74%

Constraints:
    - Total weights = 200% (leverage)
    - Each sleeve >= 10%
    - Equity sleeve <= 50%
    - Commodity sleeve <= 50%
    - Individual assets >= 0 (no shorting)

@author: Levan
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import yfinance as yf
import datetime
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

DATA_FILE = 'AssetPrices_Part2.xlsx'
FLAG_DOWNLOAD_DATA = False

START_DATE = datetime.datetime(2014, 1, 1)
END_DATE   = datetime.datetime(2025, 12, 31)

TRAIN_END  = datetime.datetime(2018, 12, 31)
TEST_START = datetime.datetime(2019, 1, 1)

FUND_TICKERS = {
    'equity':    ['VTI', 'HEDJ', 'VWO', 'QQQ', 'IWM', 'EWJ', 'INDA'],
    'bonds':     ['AGG', 'BNDX', 'TLT', 'LQD', 'HYG'],
    'tips':      ['TIP', 'STPZ'],
    'commodity': ['GLD', 'USO', 'CPER', 'SLV', 'DBA', 'UNG'],
}
FUND_LIST = (FUND_TICKERS['equity'] + FUND_TICKERS['bonds']
             + FUND_TICKERS['tips'] + FUND_TICKERS['commodity'])
N_FUND = len(FUND_LIST)

SLEEVE_INDICES = {}
_idx = 0
for sleeve, tickers in FUND_TICKERS.items():
    SLEEVE_INDICES[sleeve] = list(range(_idx, _idx + len(tickers)))
    _idx += len(tickers)

# Benchmark proxy tickers — SAME as Part 1 for consistency
BENCH_TICKERS = ['ACWI', 'AGG', 'GSG', 'TIP']

ALL_TICKERS = list(dict.fromkeys(FUND_LIST + BENCH_TICKERS))

BENCH_WEIGHTS_RAW = {
    'global_bonds':  0.7612,
    'global_equity': 0.4849,
    'commodity':     0.4066,
    'tips':          0.3474,
}
_scale = 2.0 / sum(BENCH_WEIGHTS_RAW.values())
BENCH_WEIGHTS = {k: v * _scale for k, v in BENCH_WEIGHTS_RAW.items()}

LEVERAGE      = 2.0
SLEEVE_MIN    = 0.10
EQUITY_CAP    = 0.50
COMMODITY_CAP = 0.50
LAMBDA_EWMA   = 0.94
TRADING_DAYS  = 252

# =============================================================================
# 2. UTILITY FUNCTIONS
# =============================================================================

def cov_ewma(ret_df, lamda=0.94):
    """EWMA covariance — L10 lecture code (normalized version)."""
    ret_mat = ret_df.values
    T = len(ret_df)
    S = np.cov(ret_mat.T)
    coeff = 0.0
    for i in range(1, T):
        S = lamda * S + (1 - lamda) * np.outer(ret_mat[i - 1], ret_mat[i - 1])
        coeff += (1 - lamda) * lamda ** i
    return S / coeff if coeff > 0 else S

def obj_te(w, cov_ff, cov_fb, var_b):
    """TE objective: sqrt(w'Cov_ff w - 2 w'cov_fb + var_b)."""
    te_sq = w @ cov_ff @ w - 2.0 * w @ cov_fb + var_b
    return np.sqrt(max(te_sq, 0.0))

def annualized_return(ret_series):
    cum = (1 + ret_series).prod()
    n_years = len(ret_series) / TRADING_DAYS
    return cum ** (1 / n_years) - 1 if n_years > 0 else 0.0

def annualized_vol(ret_series):
    return ret_series.std() * np.sqrt(TRADING_DAYS)

def sharpe_ratio(ret_series, rf=0.0):
    vol = annualized_vol(ret_series)
    return annualized_return(ret_series) / vol if vol > 0 else 0.0

def max_drawdown(ret_series):
    cum = (1 + ret_series).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

# =============================================================================
# 3. DATA LOADING
# =============================================================================

def download_prices(tickers, start, end, outfile):
    print(f"Downloading {len(tickers)} tickers from Yahoo Finance...")
    data = yf.download(tickers, start=start, end=end, auto_adjust=False, progress=False)
    prices = data['Adj Close'][tickers]
    prices = prices.dropna(how='all')
    with pd.ExcelWriter(outfile, engine='openpyxl') as writer:
        prices.to_excel(writer, sheet_name='Prices')
    print(f"Saved to {outfile}: {prices.shape[0]} rows, {prices.shape[1]} cols")
    return prices

def load_prices(infile):
    return pd.read_excel(infile, sheet_name='Prices', index_col=0, parse_dates=True)

if FLAG_DOWNLOAD_DATA:
    prices = download_prices(ALL_TICKERS, START_DATE, END_DATE, DATA_FILE)
else:
    prices = load_prices(DATA_FILE)

prices = prices.dropna()
returns_all = prices.pct_change().dropna()
print(f"\nData range: {returns_all.index[0].date()} to {returns_all.index[-1].date()}")
print(f"Total observations: {len(returns_all)}")

# =============================================================================
# 4. BUILD BENCHMARK RETURN SERIES (consistent with Part 1)
# =============================================================================

def build_benchmark_returns(returns):
    """Uses SAME proxy tickers as Part 1: AGG, ACWI, GSG, TIP."""
    return (BENCH_WEIGHTS['global_bonds']  * returns['AGG']
          + BENCH_WEIGHTS['global_equity'] * returns['ACWI']
          + BENCH_WEIGHTS['commodity']     * returns['GSG']
          + BENCH_WEIGHTS['tips']          * returns['TIP']).rename('Benchmark')

ret_bench = build_benchmark_returns(returns_all)
ret_fund_assets = returns_all[FUND_LIST]
ret_combined = pd.concat([ret_fund_assets, ret_bench], axis=1).dropna()

# =============================================================================
# 5. TRAIN / TEST SPLIT
# =============================================================================

ret_train = ret_combined.loc[:TRAIN_END]
ret_test  = ret_combined.loc[TEST_START:]

print(f"\nTraining: {ret_train.index[0].date()} to {ret_train.index[-1].date()} "
      f"({len(ret_train)} obs)")
print(f"Testing:  {ret_test.index[0].date()} to {ret_test.index[-1].date()} "
      f"({len(ret_test)} obs)")

# =============================================================================
# 6. COVARIANCE ESTIMATION
# =============================================================================

def compute_risk_inputs_ewma(ret_fund, ret_bench, lamda=0.94):
    combined = pd.concat([ret_fund, ret_bench], axis=1)
    combined_demean = combined - combined.mean()
    cov_full = cov_ewma(combined_demean, lamda) * TRADING_DAYS
    n = ret_fund.shape[1]
    return cov_full[:n, :n], cov_full[:n, n], cov_full[n, n]

def compute_risk_inputs_ma(ret_fund, ret_bench):
    combined = pd.concat([ret_fund, ret_bench], axis=1)
    cov_full = combined.cov().values * TRADING_DAYS
    n = ret_fund.shape[1]
    return cov_full[:n, :n], cov_full[:n, n], cov_full[n, n]

cov_ff_ewma, cov_fb_ewma, var_b_ewma = compute_risk_inputs_ewma(
    ret_train[FUND_LIST], ret_train['Benchmark'], LAMBDA_EWMA)
cov_ff_ma, cov_fb_ma, var_b_ma = compute_risk_inputs_ma(
    ret_train[FUND_LIST], ret_train['Benchmark'])

# =============================================================================
# 7. TRACKING ERROR OPTIMIZATION
# =============================================================================

def optimize_fund_weights(cov_ff, cov_fb, var_b, use_floor=True):
    n = len(FUND_LIST)
    w0 = np.ones(n) * LEVERAGE / n

    idx_eq = SLEEVE_INDICES['equity']
    idx_bd = SLEEVE_INDICES['bonds']
    idx_tp = SLEEVE_INDICES['tips']
    idx_cm = SLEEVE_INDICES['commodity']

    constraints = [
        {'type': 'eq',   'fun': lambda w: np.sum(w) - LEVERAGE},
        {'type': 'ineq', 'fun': lambda w: EQUITY_CAP    - np.sum(w[idx_eq])},
        {'type': 'ineq', 'fun': lambda w: COMMODITY_CAP  - np.sum(w[idx_cm])},
    ]
    if use_floor:
        constraints += [
            {'type': 'ineq', 'fun': lambda w: np.sum(w[idx_eq]) - SLEEVE_MIN},
            {'type': 'ineq', 'fun': lambda w: np.sum(w[idx_bd]) - SLEEVE_MIN},
            {'type': 'ineq', 'fun': lambda w: np.sum(w[idx_tp]) - SLEEVE_MIN},
            {'type': 'ineq', 'fun': lambda w: np.sum(w[idx_cm]) - SLEEVE_MIN},
        ]

    bounds = [(0.0, LEVERAGE)] * n
    result = minimize(obj_te, w0, args=(cov_ff, cov_fb, var_b),
                      method='SLSQP', bounds=bounds, constraints=constraints,
                      options={'ftol': 1e-10, 'maxiter': 10000, 'disp': False})
    if not result.success:
        print(f"WARNING: optimizer did not converge: {result.message}")
    return result.x

w_opt_ma      = optimize_fund_weights(cov_ff_ma, cov_fb_ma, var_b_ma, use_floor=True)
w_opt_ewma    = optimize_fund_weights(cov_ff_ewma, cov_fb_ewma, var_b_ewma, use_floor=True)
w_opt_nofloor = optimize_fund_weights(cov_ff_ewma, cov_fb_ewma, var_b_ewma, use_floor=False)

# =============================================================================
# 8. COMPUTE TRACKING ERRORS
# =============================================================================

def compute_te_from_returns(fund_ret, bench_ret):
    return (fund_ret - bench_ret).std() * np.sqrt(TRADING_DAYS)

def fund_return_series(weights, asset_returns):
    return (asset_returns[FUND_LIST] * weights).sum(axis=1)

fund_ret_train_ewma = fund_return_series(w_opt_ewma, ret_train)
fund_ret_test_ewma  = fund_return_series(w_opt_ewma, ret_test)
fund_ret_train_ma   = fund_return_series(w_opt_ma,   ret_train)
fund_ret_test_ma    = fund_return_series(w_opt_ma,   ret_test)

bench_train = ret_train['Benchmark']
bench_test  = ret_test['Benchmark']

te_is_ewma  = compute_te_from_returns(fund_ret_train_ewma, bench_train)
te_is_ma    = compute_te_from_returns(fund_ret_train_ma,   bench_train)
te_oos_ewma = compute_te_from_returns(fund_ret_test_ewma,  bench_test)
te_oos_ma   = compute_te_from_returns(fund_ret_test_ma,    bench_test)
te_fcst_ewma = obj_te(w_opt_ewma, cov_ff_ewma, cov_fb_ewma, var_b_ewma)
te_fcst_ma   = obj_te(w_opt_ma,   cov_ff_ma,   cov_fb_ma,   var_b_ma)

# =============================================================================
# 9. ROLLING 250-DAY TE
# =============================================================================

WINDOW = 250
fund_ret_full_ewma = fund_return_series(w_opt_ewma, ret_combined)
fund_ret_full_ma   = fund_return_series(w_opt_ma,   ret_combined)
active_full = fund_ret_full_ma - ret_combined['Benchmark']
rolling_te = active_full.rolling(WINDOW).std() * np.sqrt(TRADING_DAYS)
rolling_te_test = rolling_te.loc[TEST_START:].dropna()

# =============================================================================
# 10. SENSITIVITY ANALYSIS
# =============================================================================

split_candidates = [
    ('2016-12-31', '2017-01-01', '3yr train (2014-2016)'),
    ('2017-12-31', '2018-01-01', '4yr train (2014-2017)'),
    ('2018-12-31', '2019-01-01', '5yr train (2014-2018)'),
    ('2019-12-31', '2020-01-01', '6yr train (2014-2019)'),
    ('2020-12-31', '2021-01-01', '7yr train (2014-2020)'),
]

print("\n" + "=" * 90)
print("SENSITIVITY ANALYSIS: EFFECT OF TRAIN/TEST SPLIT POINT")
print("=" * 90)
print(f"{'Split':<28} {'Model':<8} {'Fcst TE':>10} {'IS TE':>10} "
      f"{'OOS TE':>10} {'Gap (bps)':>10}")
print("-" * 90)

sensitivity_results = []
for train_end_str, test_start_str, label in split_candidates:
    r_train_i = ret_combined.loc[:train_end_str]
    r_test_i  = ret_combined.loc[test_start_str:]
    if len(r_train_i) < 250 or len(r_test_i) < 250:
        print(f"{label:<28} {'SKIP':<8} — insufficient data")
        continue
    for model_name in ['EWMA', 'MA']:
        if model_name == 'EWMA':
            cov_ff_i, cov_fb_i, var_b_i = compute_risk_inputs_ewma(
                r_train_i[FUND_LIST], r_train_i['Benchmark'], LAMBDA_EWMA)
        else:
            cov_ff_i, cov_fb_i, var_b_i = compute_risk_inputs_ma(
                r_train_i[FUND_LIST], r_train_i['Benchmark'])
        w_i = optimize_fund_weights(cov_ff_i, cov_fb_i, var_b_i, use_floor=True)
        te_fcst_i = obj_te(w_i, cov_ff_i, cov_fb_i, var_b_i)
        te_is_i = compute_te_from_returns(fund_return_series(w_i, r_train_i), r_train_i['Benchmark'])
        te_oos_i = compute_te_from_returns(fund_return_series(w_i, r_test_i), r_test_i['Benchmark'])
        gap_i = (te_oos_i - te_fcst_i) * 10000
        print(f"{label:<28} {model_name:<8} {te_fcst_i*10000:>9.1f}  "
              f"{te_is_i*10000:>9.1f}  {te_oos_i*10000:>9.1f}  {gap_i:>+9.1f}")
        sensitivity_results.append({
            'Split': label, 'Model': model_name,
            'Forecast TE (bps)': te_fcst_i * 10000,
            'In-Sample TE (bps)': te_is_i * 10000,
            'Out-of-Sample TE (bps)': te_oos_i * 10000,
            'Gap (bps)': gap_i,
        })
print("-" * 90)

df_sensitivity = pd.DataFrame(sensitivity_results)
if len(df_sensitivity) > 0:
    best_row = df_sensitivity.loc[df_sensitivity['Gap (bps)'].abs().idxmin()]
    print(f"\nBest config (smallest |gap|):  {best_row['Split']}, "
          f"{best_row['Model']} — gap = {best_row['Gap (bps)']:+.1f} bps")
    lowest_oos = df_sensitivity.loc[df_sensitivity['Out-of-Sample TE (bps)'].idxmin()]
    print(f"Lowest OOS TE:                 {lowest_oos['Split']}, "
          f"{lowest_oos['Model']} — OOS TE = {lowest_oos['Out-of-Sample TE (bps)']:.1f} bps")

# =============================================================================
# 11. PERFORMANCE ANALYTICS
# =============================================================================

bench_asset_rets = {
    'ACWI (Global Eq)':  returns_all['ACWI'],
    'AGG (Bonds)':       returns_all['AGG'],
    'GSG (Commodities)': returns_all['GSG'],
    'TIP (TIPS)':        returns_all['TIP'],
}

all_ret_series = {
    'Optimized ETF (EWMA)': fund_ret_full_ewma,
    'Optimized ETF (MA)':   fund_ret_full_ma,
    'Benchmark':            ret_combined['Benchmark'],
}
all_ret_series.update(bench_asset_rets)
for t in FUND_LIST:
    all_ret_series[t] = returns_all[t]

stats_full = pd.DataFrame({
    name: {
        'Ann. Return':  annualized_return(s.loc[ret_combined.index]),
        'Ann. Vol':     annualized_vol(s.loc[ret_combined.index]),
        'Sharpe':       sharpe_ratio(s.loc[ret_combined.index]),
        'Max Drawdown': max_drawdown(s.loc[ret_combined.index]),
    } for name, s in all_ret_series.items()
}).T

stats_test = pd.DataFrame({
    name: {
        'Ann. Return':  annualized_return(s.loc[TEST_START:]),
        'Ann. Vol':     annualized_vol(s.loc[TEST_START:]),
        'Sharpe':       sharpe_ratio(s.loc[TEST_START:]),
        'Max Drawdown': max_drawdown(s.loc[TEST_START:]),
    } for name, s in all_ret_series.items()
}).T

# =============================================================================
# 12. PRINT SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("OPTIMIZED FUND WEIGHTS (individual assets)")
print("=" * 70)
weights_df = pd.DataFrame({
    'EWMA (10% floor)': w_opt_ewma,
    'MA (10% floor)':   w_opt_ma,
    'EWMA (no floor)':  w_opt_nofloor,
}, index=FUND_LIST)
print(weights_df.round(4))

print("\n" + "=" * 70)
print("SLEEVE-LEVEL WEIGHT COMPARISON")
print("=" * 70)
def sleeve_weights(w):
    return {
        'Equity':    np.sum(w[SLEEVE_INDICES['equity']]),
        'Bonds':     np.sum(w[SLEEVE_INDICES['bonds']]),
        'TIPS':      np.sum(w[SLEEVE_INDICES['tips']]),
        'Commodity': np.sum(w[SLEEVE_INDICES['commodity']]),
        'Total':     w.sum(),
    }
sleeve_df = pd.DataFrame({
    'EWMA': sleeve_weights(w_opt_ewma),
    'MA': sleeve_weights(w_opt_ma),
    'No floor': sleeve_weights(w_opt_nofloor),
    'Benchmark': {
        'Equity': BENCH_WEIGHTS['global_equity'],
        'Bonds': BENCH_WEIGHTS['global_bonds'],
        'TIPS': BENCH_WEIGHTS['tips'],
        'Commodity': BENCH_WEIGHTS['commodity'],
        'Total': sum(BENCH_WEIGHTS.values()),
    },
})
print(sleeve_df.round(4))

print("\n" + "=" * 70)
print("TRACKING ERROR SUMMARY (annualized)")
print("=" * 70)
te_table = pd.DataFrame({
    'Forecasted (train cov)':  [te_fcst_ma,    te_fcst_ewma],
    'Realized in-sample':      [te_is_ma,      te_is_ewma],
    'Realized out-of-sample':  [te_oos_ma,     te_oos_ewma],
    'Forecast - Realized OOS': [te_fcst_ma - te_oos_ma, te_fcst_ewma - te_oos_ewma],
}, index=['MA (primary)', 'EWMA (comparison)'])
print((te_table * 10000).round(1).astype(str) + ' bps')

print("\n" + "=" * 70)
print("PERFORMANCE COMPARISON — Full Period")
print("=" * 70)
print(stats_full[['Ann. Return', 'Ann. Vol', 'Sharpe', 'Max Drawdown']].round(4))

print("\n" + "=" * 70)
print("PERFORMANCE COMPARISON — Test Period (Out-of-Sample)")
print("=" * 70)
print(stats_test[['Ann. Return', 'Ann. Vol', 'Sharpe', 'Max Drawdown']].round(4))

print("\n" + "=" * 70)
print("OPTIMIZED WEIGHTS vs BENCHMARK SLEEVE ALLOCATION")
print("=" * 70)
print(f"Equity sleeve:    {np.sum(w_opt_ma[SLEEVE_INDICES['equity']]):.2%} "
      f"(Benchmark: {BENCH_WEIGHTS['global_equity']:.2%})")
print(f"Bond sleeve:      {np.sum(w_opt_ma[SLEEVE_INDICES['bonds']]):.2%} "
      f"(Benchmark: {BENCH_WEIGHTS['global_bonds']:.2%})")
print(f"TIPS sleeve:      {np.sum(w_opt_ma[SLEEVE_INDICES['tips']]):.2%} "
      f"(Benchmark: {BENCH_WEIGHTS['tips']:.2%})")
print(f"Commodity sleeve: {np.sum(w_opt_ma[SLEEVE_INDICES['commodity']]):.2%} "
      f"(Benchmark: {BENCH_WEIGHTS['commodity']:.2%})")

# =============================================================================
# 13. PLOTS
# =============================================================================

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    plt.style.use('default')

fig_num = 1

# ---- Figure 1: Weight comparison ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [3, 2]})
x = np.arange(N_FUND)
width = 0.25
ax1.bar(x - width, w_opt_ewma, width, label='EWMA (with floor)', color='steelblue')
ax1.bar(x,         w_opt_ma,   width, label='MA (with floor)', color='coral')
ax1.bar(x + width, w_opt_nofloor, width, label='EWMA (no floor)', color='seagreen', alpha=0.7)
ax1.set_xticks(x); ax1.set_xticklabels(FUND_LIST, fontsize=8, rotation=45, ha='right')
ax1.set_ylabel('Weight', fontsize=12); ax1.set_title('Individual Asset Weights', fontsize=13); ax1.legend(fontsize=9)

sleeves = ['Equity', 'Bonds', 'TIPS', 'Commodity']
bench_sleeve = [BENCH_WEIGHTS['global_equity'], BENCH_WEIGHTS['global_bonds'],
                BENCH_WEIGHTS['tips'], BENCH_WEIGHTS['commodity']]
ewma_sleeve = [np.sum(w_opt_ewma[SLEEVE_INDICES[s]]) for s in ['equity','bonds','tips','commodity']]
ma_sleeve   = [np.sum(w_opt_ma[SLEEVE_INDICES[s]]) for s in ['equity','bonds','tips','commodity']]
xs = np.arange(len(sleeves))
ax2.bar(xs - width, bench_sleeve, width, label='Benchmark', color='gold', edgecolor='black')
ax2.bar(xs,         ewma_sleeve,  width, label='EWMA', color='steelblue')
ax2.bar(xs + width, ma_sleeve,    width, label='MA', color='coral')
ax2.set_xticks(xs); ax2.set_xticklabels(sleeves, fontsize=11)
ax2.set_ylabel('Weight', fontsize=12); ax2.set_title('Sleeve-Level: Benchmark vs Optimized', fontsize=13); ax2.legend(fontsize=9)
plt.suptitle('Figure 1: Optimized Fund Weights vs Benchmark', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(f'fig{fig_num}_weights.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

# ---- Figure 2: Cumulative returns — ETF vs Benchmark ----
cum_fund_ewma = (1 + fund_ret_full_ewma).cumprod()
cum_fund_ma   = (1 + fund_ret_full_ma).cumprod()
cum_bench     = (1 + ret_combined['Benchmark']).cumprod()
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(cum_fund_ewma.index, cum_fund_ewma.values, label='Optimized ETF (EWMA)', linewidth=2, color='steelblue')
ax.plot(cum_fund_ma.index, cum_fund_ma.values, label='Optimized ETF (MA)', linewidth=1.5, color='coral', alpha=0.8)
ax.plot(cum_bench.index, cum_bench.values, label='Benchmark (All Weather 200%)', linewidth=2, color='gold')
ax.axvline(TEST_START, color='red', linestyle='--', alpha=0.6, label='Train/Test split')
ax.set_ylabel('Cumulative Return', fontsize=12)
ax.set_title('Figure 2: Optimized ETF vs Benchmark Cumulative Returns', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'fig{fig_num}_cumulative_vs_bench.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

# ---- Figure 3: ETF vs 4 benchmark proxy assets ----
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(cum_fund_ma.index, cum_fund_ma.values, label='Optimized ETF (MA)', linewidth=2.5, color='steelblue')
for name, color in [('ACWI', 'green'), ('AGG', 'brown'), ('GSG', 'orange'), ('TIP', 'purple')]:
    cum_asset = (1 + returns_all[name].loc[ret_combined.index]).cumprod()
    ax.plot(cum_asset.index, cum_asset.values, label=name, linewidth=1.5, color=color, alpha=0.7)
ax.axvline(TEST_START, color='red', linestyle='--', alpha=0.4)
ax.set_ylabel('Cumulative Return', fontsize=12)
ax.set_title('Figure 3: Optimized ETF vs Benchmark Asset Classes (Unlevered)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'fig{fig_num}_cumulative_vs_assets.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

# ---- Figure 4: ETF vs 20 fund assets ----
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(cum_fund_ewma.index, cum_fund_ewma.values, label='Optimized ETF', linewidth=3, color='black', zorder=5)
cmap = plt.cm.tab20(np.linspace(0, 1, N_FUND))
for i, t in enumerate(FUND_LIST):
    cum_asset = (1 + returns_all[t].loc[ret_combined.index]).cumprod()
    ax.plot(cum_asset.index, cum_asset.values, label=t, linewidth=1.0, color=cmap[i], alpha=0.7)
ax.axvline(TEST_START, color='red', linestyle='--', alpha=0.4)
ax.set_ylabel('Cumulative Return', fontsize=12)
ax.set_title('Figure 4: Optimized ETF vs Individual Fund Assets', fontsize=14, fontweight='bold')
ax.legend(fontsize=7, ncol=4, loc='upper left'); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'fig{fig_num}_cumulative_vs_fund_assets.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

# ---- Figure 5: Rolling TE ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2.2, 1]})
ax1.plot(rolling_te_test.index, rolling_te_test.values * 10000, color='steelblue', linewidth=1.5, label=f'{WINDOW}-day rolling realized TE')
ax1.axhline(te_fcst_ma * 10000, color='orange', linestyle='-', linewidth=2, label=f'Forecasted TE ({te_fcst_ma*10000:.0f} bps, MA)')
ax1.axhline(te_oos_ma * 10000, color='green', linestyle='-', linewidth=2, label=f'Full test-period TE ({te_oos_ma*10000:.0f} bps, MA)')
ax1.set_ylabel('Tracking Error (bps)', fontsize=12); ax1.set_title(f'{WINDOW}-Day Rolling TE Over Test Period', fontsize=13)
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

ax2.hist(rolling_te_test.values * 10000, bins=40, color='steelblue', edgecolor='white', alpha=0.9)
ax2.axvline(te_fcst_ma * 10000, color='orange', linestyle='-', linewidth=2, label=f'Forecast ({te_fcst_ma*10000:.0f})')
ax2.axvline(te_oos_ma * 10000, color='green', linestyle='-', linewidth=2, label=f'Realized ({te_oos_ma*10000:.0f})')
ax2.set_xlabel('Tracking Error (bps)', fontsize=11); ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Distribution of Rolling TEs', fontsize=13); ax2.legend(fontsize=9)
plt.suptitle('Figure 5: Tracking Error Model Validation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(f'fig{fig_num}_rolling_te.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

# ---- Figure 6: Sharpe ratio comparison ----
strategies_for_sharpe = ['Optimized ETF (EWMA)', 'Optimized ETF (MA)', 'Benchmark',
                         'ACWI (Global Eq)', 'AGG (Bonds)', 'GSG (Commodities)', 'TIP (TIPS)']
sharpe_full = stats_full.loc[strategies_for_sharpe, 'Sharpe']
sharpe_test = stats_test.loc[strategies_for_sharpe, 'Sharpe']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors_sharpe = ['steelblue', 'coral', 'gold', 'green', 'brown', 'orange', 'purple']

# Dynamic year ranges from actual data
full_yr = f"{ret_combined.index[0].year}–{ret_combined.index[-1].year}"
test_yr = f"{ret_test.index[0].year}–{ret_test.index[-1].year}"

ax1.barh(range(len(strategies_for_sharpe)), sharpe_full.values, color=colors_sharpe)
ax1.set_yticks(range(len(strategies_for_sharpe))); ax1.set_yticklabels(strategies_for_sharpe, fontsize=10)
ax1.set_xlabel('Sharpe Ratio', fontsize=12); ax1.set_title(f'Full Period ({full_yr})', fontsize=13); ax1.invert_yaxis()
for i, v in enumerate(sharpe_full.values): ax1.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

ax2.barh(range(len(strategies_for_sharpe)), sharpe_test.values, color=colors_sharpe)
ax2.set_yticks(range(len(strategies_for_sharpe))); ax2.set_yticklabels(strategies_for_sharpe, fontsize=10)
ax2.set_xlabel('Sharpe Ratio', fontsize=12); ax2.set_title(f'Test Period ({test_yr})', fontsize=13); ax2.invert_yaxis()
for i, v in enumerate(sharpe_test.values): ax2.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Figure 6: Sharpe Ratio Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(f'fig{fig_num}_sharpe.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

# ---- Figure 7: Correlation matrix ----
corr_train = ret_train[FUND_LIST].corr()
fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr_train.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(N_FUND)); ax.set_yticks(range(N_FUND))
ax.set_xticklabels(FUND_LIST, fontsize=8, rotation=45, ha='right'); ax.set_yticklabels(FUND_LIST, fontsize=8)
for i in range(N_FUND):
    for j in range(N_FUND):
        color = 'white' if abs(corr_train.values[i, j]) > 0.5 else 'black'
        ax.text(j, i, f'{corr_train.values[i, j]:.2f}', ha='center', va='center', fontsize=6, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Figure 7: Fund Asset Correlation Matrix (Training Period)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(f'fig{fig_num}_correlation.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

# ---- Figure 8: Sensitivity Analysis ----
if len(df_sensitivity) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    splits = df_sensitivity['Split'].unique()
    ewma_oos = df_sensitivity[df_sensitivity['Model'] == 'EWMA']['Out-of-Sample TE (bps)'].values
    ma_oos   = df_sensitivity[df_sensitivity['Model'] == 'MA']['Out-of-Sample TE (bps)'].values
    x_s = np.arange(len(splits)); bar_w = 0.35
    ax1.bar(x_s - bar_w/2, ewma_oos, bar_w, label='EWMA', color='steelblue')
    ax1.bar(x_s + bar_w/2, ma_oos,   bar_w, label='MA', color='coral')
    ax1.set_xticks(x_s); ax1.set_xticklabels([s.split('(')[1].rstrip(')') for s in splits], fontsize=9, rotation=20)
    ax1.set_ylabel('Out-of-Sample TE (bps)', fontsize=12); ax1.set_title('OOS Tracking Error by Split Point', fontsize=13)
    ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3, axis='y')

    ewma_gap = df_sensitivity[df_sensitivity['Model'] == 'EWMA']['Gap (bps)'].abs().values
    ma_gap   = df_sensitivity[df_sensitivity['Model'] == 'MA']['Gap (bps)'].abs().values
    ax2.bar(x_s - bar_w/2, ewma_gap, bar_w, label='EWMA', color='steelblue')
    ax2.bar(x_s + bar_w/2, ma_gap,   bar_w, label='MA', color='coral')
    ax2.set_xticks(x_s); ax2.set_xticklabels([s.split('(')[1].rstrip(')') for s in splits], fontsize=9, rotation=20)
    ax2.set_ylabel('|Forecast - Realized| (bps)', fontsize=12); ax2.set_title('Forecast Accuracy by Split Point', fontsize=13)
    ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3, axis='y')
    plt.suptitle('Figure 8: Sensitivity to Training Window Length', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.savefig(f'fig{fig_num}_sensitivity.png', dpi=150, bbox_inches='tight'); plt.show(); fig_num += 1

print(f"\nDone. {fig_num - 1} figures saved.")
print("Files: " + ", ".join([f"fig{i}_*.png" for i in range(1, fig_num)]))